# Chapter 23: Attribute Descriptors
*From: Fluent Python, 2nd Edition*

## TL;DR

- A **descriptor** is any class implementing `__get__`, `__set__`, or `__delete__` -- it manages attribute access on another class.
- **Overriding descriptors** (with `__set__`) intercept attribute writes on instances; **nonoverriding descriptors** (without `__set__`) can be shadowed by instance attributes.
- `__set_name__` (Python 3.6+) lets descriptors automatically know the attribute name they are assigned to, eliminating error-prone manual naming.
- Python **methods are descriptors** -- functions implement `__get__`, which is why accessing a function through an instance returns a bound method.
- The **Validated pattern** (template method) lets you build reusable, composable descriptor families for attribute validation.

## 1. The Descriptor Protocol

A descriptor is a class that implements one or more of: `__get__`, `__set__`, `__delete__`. When an instance of a descriptor class is assigned as a **class attribute** of another class, Python routes attribute access through the descriptor's methods.

Key terminology:
- **Descriptor class**: the class implementing the protocol (e.g., `Quantity`)
- **Managed class**: the class using descriptor instances as class attributes (e.g., `LineItem`)
- **Managed instance**: an instance of the managed class
- **Storage attribute**: where the actual value lives on the managed instance

In [ ]:
class Quantity:
    """A descriptor that rejects non-positive values."""

    def __init__(self, storage_name):
        self.storage_name = storage_name  # name of the attr on the instance

    def __set__(self, instance, value):
        if value > 0:
            # Store directly in instance __dict__ to avoid infinite recursion
            instance.__dict__[self.storage_name] = value
        else:
            raise ValueError(f'{self.storage_name} must be > 0')

    def __get__(self, instance, owner):
        if instance is None:
            return self  # accessed via the class, return descriptor itself
        return instance.__dict__[self.storage_name]


class LineItem:
    weight = Quantity('weight')  # descriptor instance as class attr
    price = Quantity('price')

    def __init__(self, description, weight, price):
        self.description = description
        self.weight = weight    # triggers Quantity.__set__
        self.price = price

    def subtotal(self):
        return self.weight * self.price


# Demo
raisins = LineItem('Golden raisins', 10, 6.95)
print(f"subtotal: {raisins.subtotal()}")
print(f"weight via descriptor: {raisins.weight}")
print(f"weight in __dict__: {raisins.__dict__['weight']}")

### Why store in `instance.__dict__` directly?

If `__set__` called `setattr(instance, self.storage_name, value)`, Python would route that through the descriptor's `__set__` again, causing infinite recursion. Writing directly to `instance.__dict__` bypasses the descriptor.

In [ ]:
# Try setting an invalid value
try:
    bad = LineItem('Negative test', -5, 10)
except ValueError as e:
    print(f"Caught: {e}")

# Try setting after construction
raisins = LineItem('Golden raisins', 10, 6.95)
try:
    raisins.price = -1
except ValueError as e:
    print(f"Caught on assignment: {e}")

print(f"Price still valid: {raisins.price}")

### Automatic Naming with `__set_name__`

Python 3.6 added `__set_name__(self, owner, name)` to the descriptor protocol. The interpreter calls it for each descriptor found in a class body, passing the managed class and the attribute name. This eliminates the need to pass the name manually.

In [ ]:
class Quantity:
    """Improved descriptor -- no manual storage_name needed."""

    def __set_name__(self, owner, name):
        # Called automatically by type.__new__ during class creation
        self.storage_name = name

    def __set__(self, instance, value):
        if value > 0:
            instance.__dict__[self.storage_name] = value
        else:
            raise ValueError(f'{self.storage_name} must be > 0')

    # No __get__ needed! storage_name == managed attribute name,
    # so normal instance attribute lookup returns the right value.


class LineItem:
    weight = Quantity()  # no name argument needed!
    price = Quantity()

    def __init__(self, description, weight, price):
        self.description = description
        self.weight = weight
        self.price = price

    def subtotal(self):
        return self.weight * self.price


raisins = LineItem('Golden raisins', 10, 6.95)
print(f"subtotal: {raisins.subtotal()}")

try:
    raisins.weight = -1
except ValueError as e:
    print(f"Caught: {e}")

## 2. Overriding vs Nonoverriding Descriptors

The presence or absence of `__set__` creates two fundamentally different descriptor behaviors:

| Type | Also called | Has `__set__`? | Instance attr shadows it? |
|------|------------|-----------------|--------------------------|
| **Overriding** | data descriptor, enforced descriptor | Yes | No -- descriptor always wins |
| **Nonoverriding** | non-data descriptor, shadowable descriptor | No | Yes -- instance attr hides it |

This distinction matters because Python's attribute lookup works differently for reads vs writes:
- **Read**: checks instance `__dict__` first, then class (unless an overriding descriptor intercepts)
- **Write**: normally creates/updates in instance `__dict__` (unless an overriding descriptor intercepts)

In [ ]:
class Overriding:
    """Descriptor WITH __set__ -- always intercepts writes."""

    def __get__(self, instance, owner):
        print(f"  Overriding.__get__(instance={instance!r})")
        return 42

    def __set__(self, instance, value):
        print(f"  Overriding.__set__(instance={instance!r}, value={value!r})")


class NonOverriding:
    """Descriptor WITHOUT __set__ -- can be shadowed."""

    def __get__(self, instance, owner):
        print(f"  NonOverriding.__get__(instance={instance!r})")
        return 99


class Managed:
    over = Overriding()
    non_over = NonOverriding()


obj = Managed()

print("=== Overriding descriptor ===")
print("Read obj.over:")
val = obj.over

print("\nWrite obj.over = 7:")
obj.over = 7

print("\nWrite directly to __dict__:")
obj.__dict__['over'] = 8
print(f"obj.__dict__['over'] = {obj.__dict__['over']}")

print("\nRead obj.over again (descriptor STILL wins):")
val = obj.over

In [ ]:
print("=== Nonoverriding descriptor ===")
obj2 = Managed()

print("Read obj2.non_over:")
val = obj2.non_over

print("\nWrite obj2.non_over = 7 (creates instance attr, shadows descriptor):")
obj2.non_over = 7
print(f"obj2.non_over = {obj2.non_over}  (instance attr, not descriptor)")

print("\nDelete instance attr, descriptor reappears:")
del obj2.non_over
val = obj2.non_over

### Key insight

An **overriding descriptor** with `__set__` takes control even when you write to `instance.__dict__` directly and then read back. The descriptor's `__get__` always intercepts the read.

A **nonoverriding descriptor** steps aside the moment an instance attribute of the same name exists. That's exactly how `@functools.cached_property` works: the first access computes the value via `__get__`, stores it as an instance attribute, and subsequent accesses find the cached instance attribute without ever calling the descriptor again.

## 3. The Validated Descriptor Pattern

When you need multiple descriptor types sharing common logic, use the **template method pattern**: an abstract base descriptor defines `__set__` which delegates to an abstract `validate()` method. Each concrete subclass implements only its validation rule.

In [ ]:
import abc


class Validated(abc.ABC):
    """Abstract descriptor that delegates validation to subclasses."""

    def __set_name__(self, owner, name):
        self.storage_name = name

    def __set__(self, instance, value):
        # Template method: validate, then store
        value = self.validate(self.storage_name, value)
        instance.__dict__[self.storage_name] = value

    @abc.abstractmethod
    def validate(self, name, value):
        """Return validated value or raise ValueError."""


class Quantity(Validated):
    """A number greater than zero."""

    def validate(self, name, value):
        if value <= 0:
            raise ValueError(f'{name} must be > 0')
        return value


class NonBlank(Validated):
    """A string with at least one non-space character."""

    def validate(self, name, value):
        value = value.strip()
        if not value:
            raise ValueError(f'{name} cannot be blank')
        return value  # returns cleaned (stripped) value


class LineItem:
    description = NonBlank()
    weight = Quantity()
    price = Quantity()

    def __init__(self, description, weight, price):
        self.description = description
        self.weight = weight
        self.price = price

    def subtotal(self):
        return self.weight * self.price


# Valid item
nuts = LineItem('  Macadamia nuts  ', 5, 11.99)
print(f"description={nuts.description!r} (stripped!)")
print(f"subtotal={nuts.subtotal()}")

# Invalid: blank description
try:
    LineItem('', 1, 1)
except ValueError as e:
    print(f"Blank desc: {e}")

# Invalid: negative weight
try:
    LineItem('Almonds', -3, 5)
except ValueError as e:
    print(f"Negative weight: {e}")

The beauty of this pattern: adding a new descriptor type (e.g., `PositiveInteger`, `EmailAddress`) requires only a new subclass with a `validate()` method. The storage, naming, and `__set__` plumbing is inherited.

## 4. Methods Are Descriptors

Every Python function has a `__get__` method but no `__set__`, making functions **nonoverriding descriptors**. When a function is a class attribute and you access it through an instance, `__get__` returns a **bound method** -- a callable that automatically passes the instance as `self`.

In [ ]:
import collections


class Text(collections.UserString):

    def __repr__(self):
        return f'Text({self.data!r})'

    def reverse(self):
        return self[::-1]


word = Text('forward')

# Access through instance -> bound method
bound = word.reverse
print(f"word.reverse: {bound}")
print(f"type: {type(bound)}")

# Access through class -> plain function
func = Text.reverse
print(f"\nText.reverse: {func}")
print(f"type: {type(func)}")

# Both work, but differently
print(f"\nword.reverse(): {word.reverse()}")
print(f"Text.reverse(word): {Text.reverse(word)}")

In [ ]:
# Under the hood: function.__get__ creates the bound method

word = Text('forward')

# Manually calling __get__ on the function
bound_method = Text.reverse.__get__(word)
print(f"Manually created bound method: {bound_method}")
print(f"Calling it: {bound_method()}")

# Bound method internals
print(f"\n__self__: {bound_method.__self__}")
print(f"__func__: {bound_method.__func__}")
print(f"__func__ is Text.reverse: {bound_method.__func__ is Text.reverse}")

# When accessed via class (instance=None), __get__ returns the function itself
unbound = Text.reverse.__get__(None, Text)
print(f"\n__get__(None, Text): {unbound}")
print(f"Is same function: {unbound is Text.reverse}")

### Why this matters

This mechanism is how Python implements the implicit `self` parameter. When you write `word.reverse()`, Python does:
1. Look up `reverse` -- finds it as a class attribute (a function)
2. Function is a descriptor, so call `function.__get__(word, Text)`
3. That returns a bound method wrapping the function + the instance
4. Call the bound method, which calls `function(word)`

Since functions only have `__get__` (no `__set__`), they are **nonoverriding** -- you can shadow a method by assigning to an instance attribute: `obj.method = 7` makes `obj.method` return `7` instead of the bound method.

## 5. Descriptor Usage Tips

Practical guidelines for working with descriptors:

In [ ]:
# Tip 1: Use property to keep it simple
# property is an overriding descriptor -- easiest way to add managed access

import math

class Circle:
    def __init__(self, radius):
        self._radius = radius

    @property
    def radius(self):
        return self._radius

    @radius.setter
    def radius(self, value):
        if value <= 0:
            raise ValueError("radius must be positive")
        self._radius = value

    @property
    def area(self):
        """Read-only: property without setter raises AttributeError on write."""
        return math.pi * self._radius ** 2


c = Circle(5)
print(f"radius={c.radius}, area={c.area:.2f}")

try:
    c.area = 100  # read-only property
except AttributeError as e:
    print(f"Can't set area: {e}")

In [ ]:
# Tip 2: Read-only descriptors MUST implement __set__
# Otherwise instance attributes shadow them

class ReadOnlyDescriptor:
    """A descriptor that only implements __get__ -- BROKEN for read-only!"""
    def __get__(self, instance, owner):
        if instance is None:
            return self
        return "I am read-only"


class BrokenReadOnly:
    attr = ReadOnlyDescriptor()


obj = BrokenReadOnly()
print(f"Before: {obj.attr}")

# This SHADOWS the descriptor -- not what we want!
obj.attr = "overwritten!"
print(f"After:  {obj.attr}  (descriptor is gone!)")


# CORRECT approach: implement __set__ that raises
class ProperReadOnly:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        return "I am truly read-only"

    def __set__(self, instance, value):
        raise AttributeError("read-only attribute")


class CorrectReadOnly:
    attr = ProperReadOnly()


obj2 = CorrectReadOnly()
print(f"\nRead: {obj2.attr}")
try:
    obj2.attr = "nope"
except AttributeError as e:
    print(f"Blocked: {e}")

In [ ]:
# Tip 3: Validation-only descriptors can skip __get__
# When storage_name == attribute name, normal lookup finds the instance attr

class PositiveNumber:
    def __set_name__(self, owner, name):
        self.storage_name = name

    def __set__(self, instance, value):
        if not isinstance(value, (int, float)) or value <= 0:
            raise ValueError(f"{self.storage_name} must be a positive number")
        instance.__dict__[self.storage_name] = value
    # No __get__ -- reading goes straight to instance.__dict__


class Product:
    price = PositiveNumber()
    quantity = PositiveNumber()

    def __init__(self, name, price, quantity):
        self.name = name
        self.price = price
        self.quantity = quantity


p = Product("Widget", 9.99, 100)
print(f"{p.name}: ${p.price} x {p.quantity}")
# Fast reads -- no descriptor overhead on __get__

In [ ]:
# Tip 4: Caching with __get__-only (nonoverriding) descriptor
# First access computes, stores in instance __dict__, future reads skip descriptor

class CachedProperty:
    """Simplified version of functools.cached_property."""

    def __init__(self, func):
        self.func = func
        self.attr_name = func.__name__

    def __get__(self, instance, owner):
        if instance is None:
            return self
        # Compute value and store as instance attribute
        value = self.func(instance)
        instance.__dict__[self.attr_name] = value  # shadows this descriptor
        print(f"  (computed and cached {self.attr_name})")
        return value


class DataProcessor:
    def __init__(self, data):
        self.data = data

    @CachedProperty
    def result(self):
        """Expensive computation, done only once."""
        return sum(x ** 2 for x in self.data)


proc = DataProcessor(range(1000))
print("First access:")
print(f"  result = {proc.result}")

print("Second access (cached -- no descriptor call):")
print(f"  result = {proc.result}")

print(f"\nStored in __dict__: {'result' in proc.__dict__}")

## Try It Yourself

In [ ]:
# Exercise 1: Create a RangeValidator descriptor
# It should accept a min and max value in __init__
# and reject values outside [min, max] in __set__.
# Use __set_name__ for automatic naming.

class RangeValidator:
    # YOUR CODE HERE
    pass

# Uncomment to test:
# class Config:
#     temperature = RangeValidator(min_val=-40, max_val=150)
#     humidity = RangeValidator(min_val=0, max_val=100)
#
# c = Config()
# c.temperature = 72
# c.humidity = 45
# print(f"temp={c.temperature}, humidity={c.humidity}")
# c.temperature = 200  # Expected: ValueError

In [ ]:
# Exercise 2: Prove that methods are nonoverriding descriptors
# Create a class with a method, then shadow it with an instance attribute.
# Show that the method is gone for that instance but still works for others.

class Dog:
    def speak(self):
        return "Woof!"

# YOUR CODE HERE
# Create two Dog instances
# Shadow speak on one of them
# Show the other still works
# Expected: one dog says something else, the other still Woofs

In [ ]:
# Exercise 3: Build a TypeChecked descriptor
# that enforces a specific type on assignment.
# Hint: accept the expected type in __init__.

class TypeChecked:
    # YOUR CODE HERE
    pass

# Uncomment to test:
# class Person:
#     name = TypeChecked(str)
#     age = TypeChecked(int)
#
# p = Person()
# p.name = "Alice"
# p.age = 30
# print(f"{p.name}, age {p.age}")
# p.age = "thirty"  # Expected: TypeError

## Key Takeaways

1. A descriptor is any class with `__get__`, `__set__`, or `__delete__` -- it manages attribute access when used as a class attribute.
2. Always store managed values in `instance.__dict__`, never in the descriptor itself (descriptors are shared across all instances).
3. Use `__set_name__` (Python 3.6+) to automatically capture the attribute name -- never require users to repeat it.
4. **Overriding descriptors** (`__set__` present) always intercept attribute writes; **nonoverriding descriptors** can be shadowed by instance attributes.
5. Python methods work because functions are nonoverriding descriptors whose `__get__` returns bound methods.
6. For simple cases use `property`; write custom descriptors when you need to reuse the same validation/logic across multiple attributes or classes.
7. The **Validated pattern** (template method + ABC) is the idiomatic way to build families of related descriptors with shared plumbing.

## See Also

- [[descriptor-protocol]] -- full details on `__get__`, `__set__`, `__delete__`, and `__set_name__`
- [[overriding-vs-nonoverriding-descriptors]] -- deep dive into shadowing rules
- [[validated-descriptor-pattern]] -- template method for reusable validation
- [[methods-are-descriptors]] -- how functions become bound methods
- [[descriptor-usage-tips]] -- practical guidelines for when to use what
- [Descriptor HowTo Guide](https://docs.python.org/3/howto/descriptor.html) -- Raymond Hettinger's official guide
- [Data Model: Descriptors](https://docs.python.org/3/reference/datamodel.html#descriptors) -- Python language reference